In [1]:
import tensorflow as tf
# tf.config.experimental.set_memory_growth()
import os, yaml, h5py

from msfm.utils import files, input_output
from deep_lss.utils import configuration
from deep_lss.models.base_model import BaseModel
from deep_lss.nets import NETWORKS

24-04-25 02:29:37   imports.py INF   Setting up healpy to run on 32 CPUs 


In [2]:
n_output = 5
n_z_bins = 8
loss_func = "delta"
n_side = 512

msfm_conf = files.load_config("/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v8/linear_bias.yaml")
dlss_conf = input_output.read_yaml("/global/homes/a/athomsen/y3-deep-lss/configs/v8/combined/linear_bias/dlss_config.yaml")

data_vec_pix, _, _, _ = files.load_pixel_file(msfm_conf)

batch_size = 15 * 4
test_batch = tf.random.uniform((batch_size, len(data_vec_pix), n_z_bins))

24-04-25 02:29:44     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_v11_512.h5 


# vanilla

In [3]:
net_conf = {
    "network" : {
        "name": "resnet",
        "n_neighbors": 20,
        "max_checkpoints": 10,
        "save_second_to_last_layer": False,
        "kwargs": {
            "base_channels": 32,
            # depth
            "downsampling_layers": 3,
            "cheby_layers": 2,
            "residual_layers": 5,
            # misc
            "poly_degree": 5,
            "dropout_rate": None,
            "second_to_last_features": None,
        }
    }
}

network = NETWORKS[net_conf["network"]["name"]](
    out_features=n_output, **net_conf["network"]["kwargs"], norm_kwargs={},
).get_layers()

model = BaseModel(
    network=network,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=net_conf["network"]["n_neighbors"],
    input_shape=(None, len(data_vec_pix), n_z_bins),
    max_batch_size=batch_size,
    checkpoint_dir=None,
    restore_checkpoint=False,
    strategy=None,
)

model.tf_call(test_batch);

24-04-25 02:29:45    resnet.py WAR   No smoothing layer is included in the network 
24-04-25 02:29:45 regression_h INF   Using a dense regression head 
24-04-25 02:29:45 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 32.0, the input with nside 512 will be transformed to 16 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Model: "healpy_gcnn_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 healpy_pseudo_conv (HealpyP  (None, 116224, 32)       1056      
 seudoConv)                                                      
                                                                 
 healpy_pseudo_conv_1 (Healp  (None, 29056, 64)        8256      
 yPseudoConv)                                                    
                                                                 
 healpy_pseudo_conv_2 (Healp  (None, 7264, 

In [4]:
%%timeit
model.tf_call(test_batch)

194 ms ± 15.2 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


# deepeer

In [5]:
net_conf = {
    "network" : {
        "name": "resnet",
        "n_neighbors": 20,
        "max_checkpoints": 10,
        "save_second_to_last_layer": False,
        "kwargs": {
            "base_channels": 32,
            # depth
            "downsampling_layers": 3,
            "cheby_layers": 2,
            "residual_layers": 10,
            # misc
            "poly_degree": 5,
            "dropout_rate": None,
            "second_to_last_features": None,
        }
    }
}

network = NETWORKS[net_conf["network"]["name"]](
    out_features=n_output, **net_conf["network"]["kwargs"], norm_kwargs={},
).get_layers()

model = BaseModel(
    network=network,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=net_conf["network"]["n_neighbors"],
    input_shape=(None, len(data_vec_pix), n_z_bins),
    max_batch_size=batch_size,
    checkpoint_dir=None,
    restore_checkpoint=False,
    strategy=None,
)

model.tf_call(test_batch);

24-04-25 02:30:06    resnet.py WAR   No smoothing layer is included in the network 
24-04-25 02:30:06 regression_h INF   Using a dense regression head 
24-04-25 02:30:06 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 32.0, the input with nside 512 will be transformed to 16 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Model: "healpy_gcnn_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 healpy_pseudo_conv_5 (Healp  (None, 116224, 32)       1056      
 yPseudoConv)                                                    
                                                                 
 healpy_pseudo_conv_6 (Healp  (None, 29056, 64)        8256      
 yPseudoConv)                                                    
                                                                 
 healpy_pseudo_conv_7 (Healp  (None, 7264, 

In [6]:
%%timeit
model.tf_call(test_batch)

270 ms ± 21.8 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


# higher order

In [7]:
net_conf = {
    "network" : {
        "name": "resnet",
        "n_neighbors": 20,
        "max_checkpoints": 10,
        "save_second_to_last_layer": False,
        "kwargs": {
            "base_channels": 32,
            # depth
            "downsampling_layers": 3,
            "cheby_layers": 2,
            "residual_layers": 5,
            # misc
            "poly_degree": 7,
            "dropout_rate": None,
            "second_to_last_features": None,
        }
    }
}

network = NETWORKS[net_conf["network"]["name"]](
    out_features=n_output, **net_conf["network"]["kwargs"], norm_kwargs={},
).get_layers()

model = BaseModel(
    network=network,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=net_conf["network"]["n_neighbors"],
    input_shape=(None, len(data_vec_pix), n_z_bins),
    max_batch_size=batch_size,
    checkpoint_dir=None,
    restore_checkpoint=False,
    strategy=None,
)

model.tf_call(test_batch);

24-04-25 02:30:31    resnet.py WAR   No smoothing layer is included in the network 
24-04-25 02:30:31 regression_h INF   Using a dense regression head 
24-04-25 02:30:31 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 32.0, the input with nside 512 will be transformed to 16 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Model: "healpy_gcnn_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 healpy_pseudo_conv_10 (Heal  (None, 116224, 32)       1056      
 pyPseudoConv)                                                   
                                                                 
 healpy_pseudo_conv_11 (Heal  (None, 29056, 64)        8256      
 pyPseudoConv)                                                   
                                                                 
 healpy_pseudo_conv_12 (Heal  (None, 7264, 

In [8]:
%%timeit
model.tf_call(test_batch)

277 ms ± 115 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


# more neighbors

In [9]:
net_conf = {
    "network" : {
        "name": "resnet",
        "n_neighbors": 40,
        "max_checkpoints": 10,
        "save_second_to_last_layer": False,
        "kwargs": {
            "base_channels": 32,
            # depth
            "downsampling_layers": 3,
            "cheby_layers": 2,
            "residual_layers": 5,
            # misc
            "poly_degree": 5,
            "dropout_rate": None,
            "second_to_last_features": None,
        }
    }
}

network = NETWORKS[net_conf["network"]["name"]](
    out_features=n_output, **net_conf["network"]["kwargs"], norm_kwargs={},
).get_layers()

model = BaseModel(
    network=network,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=net_conf["network"]["n_neighbors"],
    input_shape=(None, len(data_vec_pix), n_z_bins),
    max_batch_size=batch_size,
    checkpoint_dir=None,
    restore_checkpoint=False,
    strategy=None,
)

model.tf_call(test_batch);

24-04-25 02:30:35    resnet.py WAR   No smoothing layer is included in the network 
24-04-25 02:30:35 regression_h INF   Using a dense regression head 
24-04-25 02:30:35 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 32.0, the input with nside 512 will be transformed to 16 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Tracing... Due to tensor size, tf.sparse.sparse_dense_matmul is executed over 2 splits. Beware of the resulting performance penalty.
Model: "healpy_gcnn_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 healpy_pseudo_conv_15 (Heal  (None, 116224, 32)       1056      
 pyPseudoConv)                                                   
                                                                 
 healpy_pseudo_conv_16 (Heal  (None, 29056, 64)        8256      
 pyPseudoConv)                            

In [10]:
%%timeit
model.tf_call(test_batch)

299 ms ± 167 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
net_conf = {
    "network" : {
        "name": "resnet",
        "n_neighbors": 60,
        "max_checkpoints": 10,
        "save_second_to_last_layer": False,
        "kwargs": {
            "base_channels": 32,
            # depth
            "downsampling_layers": 3,
            "cheby_layers": 2,
            "residual_layers": 5,
            # misc
            "poly_degree": 5,
            "dropout_rate": None,
            "second_to_last_features": None,
        }
    }
}

network = NETWORKS[net_conf["network"]["name"]](
    out_features=n_output, **net_conf["network"]["kwargs"], norm_kwargs={},
).get_layers()

model = BaseModel(
    network=network,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=net_conf["network"]["n_neighbors"],
    input_shape=(None, len(data_vec_pix), n_z_bins),
    max_batch_size=batch_size,
    checkpoint_dir=None,
    restore_checkpoint=False,
    strategy=None,
)

model.tf_call(test_batch);

24-04-25 02:32:30    resnet.py WAR   No smoothing layer is included in the network 
24-04-25 02:32:30 regression_h INF   Using a dense regression head 
24-04-25 02:32:30 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 32.0, the input with nside 512 will be transformed to 16 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Model: "healpy_gcnn_9"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 healpy_pseudo_conv_20 (Heal  (None, 116224, 32)       1056      
 pyPseudoConv)                                                   
                                                                 
 healpy_pseudo_conv_21 (Heal  (None, 29056, 64)        8256      
 pyPseudoConv)                                                   
                                                                 
 healpy_pseudo_conv_22 (Heal  (None, 7264, 

In [12]:
%%timeit
model.tf_call(test_batch)

402 ms ± 265 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
